# Gravity ML — Colab GPU Training

1. Export CSV locally: `PYTHONPATH=. python3 -m gravity_api.jobs.export_training_csv --mode scored --sport cfb --out cfb_scored.csv`
2. Upload `cfb_scored.csv` below (or pull from Drive/S3).
3. Train ValueNet / QualityNet; upload `model.pkl` + `feature_manifest.json` to S3 / Railway bundle path.

In [ ]:
!pip -q install pandas scikit-learn torch joblib

In [ ]:
import math
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Upload CSV in Colab: Runtime → Upload, or mount Google Drive
CSV_PATH = 'cfb_scored.csv'  # change to your file
OBJECTIVE = 'value'  # value | quality
SPORT = 'cfb'

In [ ]:
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

In [ ]:
META = {'entity_id', 'entity_type', 'sport', 'as_of', 'name', 'school', 'position', 'conference', 'class_year', 'model_version', 'partnership_top_brand_1'}
TARGET_COL = 'target_log_nil_usd' if OBJECTIVE == 'value' else 'target_quality'

if TARGET_COL not in df.columns:
    raise ValueError(f'Missing {TARGET_COL} — use scored export or add labels')

feature_cols = [
    c for c in df.columns
    if c not in META and not c.startswith('target_') and not c.startswith('label_') and not c.startswith('dollar_')
]
feature_cols = [c for c in feature_cols if df[c].dtype != 'object']

work = df.dropna(subset=[TARGET_COL]).copy()
X = work[feature_cols].fillna(0).astype(float).values
y = work[TARGET_COL].astype(float).values
print(f'Features: {len(feature_cols)}, rows: {len(work)}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, shuffle=False)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

class GravityMLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = GravityMLP(X_train_s.shape[1]).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.HuberLoss()

Xt = torch.tensor(X_train_s, dtype=torch.float32, device=device)
yt = torch.tensor(y_train, dtype=torch.float32, device=device)

for epoch in range(200):
    model.train()
    opt.zero_grad()
    pred = model(Xt)
    loss = loss_fn(pred, yt)
    loss.backward()
    opt.step()
    if epoch % 50 == 0:
        print(f'epoch {epoch} loss={loss.item():.4f}')

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(torch.tensor(X_test_s, dtype=torch.float32, device=device)).cpu().numpy()
mae = np.mean(np.abs(pred - y_test))
print(f'Test MAE: {mae:.4f}')

out_dir = Path(f'gravity_athlete_{SPORT}_{OBJECTIVE}_v1/1.0.0-colab')
out_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), out_dir / 'model.pt')
pickle.dump(scaler, open(out_dir / 'normalizer.pkl', 'wb'))
(out_dir / 'feature_manifest.json').write_text(json.dumps({'feature_names': feature_cols}, indent=2))
(out_dir / 'metrics.json').write_text(json.dumps({'test_mae': float(mae)}, indent=2))
print(f'Saved bundle to {out_dir} — zip and upload to S3 / Railway MODEL_BUNDLE_ROOT')